In [4]:
# -*- coding: utf-8 -*-
"""
Gradle scan (all *.gradle / *.gradle.kts) + AT emptiness check, with robust diagnostics.
- Reads CSV with a 'full_name' (or fallback columns) and writes AT_check, AT_empty, Build_check, Build_reason.
- Never sys.exit; prints helpful errors instead.
"""

import os
import re
import shutil
import datetime as dt
from pathlib import Path
from typing import Dict, List, Tuple, Set

import pandas as pd

# ---------- CONFIG ----------
SUPPORT_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\support")
INPUT_CSV_NAME = "Check.csv"           # set to None to auto-detect
CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files")
TESTS_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Test_Files")
MAKE_BACKUP = True
# ---------------------------

ANDROID_TEST_SOURCE_PATTERNS = [
    r"@RunWith\s*\(\s*AndroidJUnit4(::class)?\s*\)",
    r"\bAndroidJUnit4\b",
    r"\bandroidx\.test\.",
    r"\bandroid\.test\.",
    r"\bInstrumentationRegistry\b",
    r"\bApplicationProvider\b",
    r"\bEspresso\b",
]

BUILD_SIGNAL_PATTERNS = {
    "testInstrumentationRunner": r"\btestInstrumentationRunner\b\s*(=|\s)\s*['\"][^'\"]+['\"]?",
    "androidTestDependency_str": r"\bandroidTest(Implementation|Api|CompileOnly|RuntimeOnly)\s*\(?\s*['\"][^'\"]+['\"]",
    "androidTestDependency_alias": r"\bandroidTest(Implementation|Api|CompileOnly|RuntimeOnly)\s*\(\s*[a-zA-Z0-9_.]+\s*\)",
    "androidTestDependency_add": r"\badd\s*\(\s*['\"]androidTest(Implementation|Api|CompileOnly|RuntimeOnly)['\"]\s*,\s*[^)]+\)",
    "androidx_test_dependency": r"['\"][^'\"]*androidx\.test[^'\"]*['\"]",
    "espresso_dependency": r"['\"][^'\"]*espresso[^'\"]*['\"]",
    "uiautomator_dependency": r"['\"][^'\"]*uiautomator[^'\"]*['\"]",
    "benchmark_dependency": r"['\"][^'\"]*androidx\.benchmark[^'\"]*['\"]",
    "orchestrator_dependency": r"['\"][^'\"]*androidx\.test:orchestrator[^'\"]*['\"]",
    "managed_devices": r"\btestOptions\s*\{[^}]*managedDevices\b|testOptions\.managedDevices",
    "useOrchestrator": r"\buseOrchestrator\s*=\s*true|\buseOrchestrator\s+true",
    "connectedAndroidTest_task": r"\bconnectedAndroidTest\b",
}

def safe_read_text(path: Path) -> str:
    for enc in ("utf-8", "latin-1"):
        try:
            return path.read_text(encoding=enc, errors="ignore")
        except Exception:
            continue
    return ""

def is_gradle_script(p: Path) -> bool:
    # Accept both .gradle and .gradle.kts (Path.suffix gives only the last; use name endswith)
    return p.is_file() and (p.name.endswith(".gradle") or p.name.endswith(".gradle.kts"))

def repo_key_from_filename(fname: str) -> str:
    base = Path(fname).name
    key = base.split("__", 1)[0] if "__" in base else Path(base).stem
    return key.strip().lower()

def index_files_by_repo(root: Path, only_gradle: bool) -> Dict[str, List[Path]]:
    idx: Dict[str, List[Path]] = {}
    if not root.exists():
        return idx
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        if only_gradle and not is_gradle_script(p):
            continue
        key = repo_key_from_filename(p.name)
        idx.setdefault(key, []).append(p)
    return idx

def detect_android_test_in_sources(paths: List[Path]) -> Tuple[bool, bool]:
    if not paths:
        return (False, False)
    compiled = [re.compile(p, re.IGNORECASE | re.DOTALL) for p in ANDROID_TEST_SOURCE_PATTERNS]
    has_any = True
    has_instru = False
    for f in paths:
        txt = safe_read_text(f)
        if txt and any(pat.search(txt) for pat in compiled):
            has_instru = True
            break
    return (has_any, has_instru)

def detect_instrumentation_build_signals(paths: List[Path]) -> Tuple[bool, Set[str]]:
    if not paths:
        return (False, set())
    compiled = {k: re.compile(v, re.IGNORECASE | re.DOTALL) for k, v in BUILD_SIGNAL_PATTERNS.items()}
    reasons: Set[str] = set()
    for f in paths:
        txt = safe_read_text(f)
        if not txt:
            continue
        for name, pat in compiled.items():
            if pat.search(txt):
                reasons.add(name)
    return (len(reasons) > 0, reasons)

def pick_input_csv() -> Path:
    if not SUPPORT_DIR.exists():
        raise FileNotFoundError(f"SUPPORT_DIR does not exist: {SUPPORT_DIR}")
    if INPUT_CSV_NAME:
        cand = SUPPORT_DIR / INPUT_CSV_NAME
        if cand.exists():
            return cand
        # fall through to auto if specified name missing
        print(f"[WARN] '{INPUT_CSV_NAME}' not found in {SUPPORT_DIR}. Auto-detecting...")
    csvs = sorted(SUPPORT_DIR.glob("*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in: {SUPPORT_DIR}")
    # Prefer Check.csv if present among many, else most recent
    for p in csvs:
        if p.name.lower() == "check.csv":
            return p
    return csvs[0]

def get_full_name_column(df: pd.DataFrame) -> str:
    # Accept common fallbacks
    candidates = ["full_name", "repo", "owner_repo"]
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Required column not found. Tried {candidates}. Available columns: {list(df.columns)}")

def main():
    try:
        input_csv = pick_input_csv()
        print(f"[INFO] Using CSV: {input_csv}")

        if MAKE_BACKUP:
            ts = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
            backup = input_csv.with_name(f"{input_csv.stem}__backup_{ts}{input_csv.suffix}")
            shutil.copy2(input_csv, backup)
            print(f"[INFO] Backup written: {backup}")

        # Load
        df = pd.read_csv(input_csv)
        name_col = get_full_name_column(df)

        # Index
        if not CONFIG_DIR.exists():
            print(f"[WARN] CONFIG_DIR does not exist: {CONFIG_DIR}")
        if not TESTS_DIR.exists():
            print(f"[WARN] TESTS_DIR does not exist: {TESTS_DIR}")

        print("[INFO] Indexing Gradle scripts (*.gradle / *.gradle.kts) in All_Config_Files ...")
        config_idx = index_files_by_repo(CONFIG_DIR, only_gradle=True)
        print(f"[INFO] Repos with Gradle scripts: {len(config_idx)}")

        print("[INFO] Indexing all saved test files in All_Test_Files ...")
        tests_idx = index_files_by_repo(TESTS_DIR, only_gradle=False)
        print(f"[INFO] Repos with any saved tests: {len(tests_idx)}")

        at_results: List[str] = []
        at_empty_flags: List[str] = []
        build_results: List[str] = []
        build_reasons_col: List[str] = []

        for _, row in df.iterrows():
            repo = str(row[name_col]).strip().lower()

            # AT
            test_paths = tests_idx.get(repo, [])
            has_any_test_files, has_androidTest = detect_android_test_in_sources(test_paths)
            if not has_any_test_files:
                at_check = "no_test_files_found"
            else:
                at_check = "has_androidTest" if has_androidTest else "only_unit_or_unknown_tests"
            at_results.append(at_check)
            at_empty_flags.append("yes" if not has_androidTest else "no")

            # Build
            cfg_paths = config_idx.get(repo, [])
            has_build_sig, reasons = detect_instrumentation_build_signals(cfg_paths)
            build_results.append("yes" if has_build_sig else "no")
            build_reasons_col.append("; ".join(sorted(reasons)) if reasons else "")

        df["AT_check"] = at_results
        df["AT_empty"] = at_empty_flags
        df["Build_check"] = build_results
        df["Build_reason"] = build_reasons_col

        df.to_csv(input_csv, index=False, encoding="utf-8")
        print(f"[DONE] Updated CSV saved: {input_csv}")

    except Exception as e:
        # Friendly diagnostics for notebooks/terminals; no sys.exit
        print("\n[ERROR] Scan aborted.")
        print(f"Reason: {type(e).__name__}: {e}")
        print("Tips:")
        print(" • Ensure SUPPORT_DIR, CONFIG_DIR, TESTS_DIR exist.")
        print(" • Make sure the CSV has a 'full_name' column (or 'repo' / 'owner_repo').")
        print(" • If multiple CSVs exist, this picks the most recent (or Check.csv if present).")
        try:
            import traceback
            traceback.print_exc()
        except Exception:
            pass

if __name__ == "__main__":
    main()


[INFO] Using CSV: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\support\Check.csv
[INFO] Backup written: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\support\Check__backup_20250825_113622.csv
[INFO] Indexing Gradle scripts (*.gradle / *.gradle.kts) in All_Config_Files ...
[INFO] Repos with Gradle scripts: 4394
[INFO] Indexing all saved test files in All_Test_Files ...
[INFO] Repos with any saved tests: 1770
[DONE] Updated CSV saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\support\Check.csv


In [6]:
# check the ymal and scripts for raw text check to find any instru testing signal

# -*- coding: utf-8 -*-
"""
Scan raw text of YAML + support scripts for instrumentation-testing signals.
Outputs a per-file CSV with: full_name, File_name, ci_platform_guess, detection (yes/no), detection_reason.

- Reads support CSV (must contain 'full_name' or a fallback) to limit repos scanned.
- Scans All_Config_Files for *.yml, *.yaml, and support scripts (*.sh, *.bash, *.ps1, *.bat, *.cmd, *.py).
- EXCLUDES Gradle build files (*.gradle, *.gradle.kts).
- Does not compute test presence or build checks; this is YAML/support-scripts only.
"""

import os
import re
import csv
import shutil
import datetime as dt
from pathlib import Path
from typing import Dict, List, Tuple, Set

import pandas as pd

# ---------- CONFIG ----------
SUPPORT_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\support")
INPUT_CSV_NAME = "Check_Y1_B0_A0.csv"                       # set to None to auto-detect most recent CSV
CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files")
MAKE_BACKUP = False                                # we don't overwrite the input CSV in this script
OUTPUT_CSV = SUPPORT_DIR / "Check_Y1_B0_A0_output.csv"
MAX_BYTES = 2_000_000                              # skip unusually large files (> ~2 MB) for safety
# ---------------------------

# ---- File filters ----
YAML_EXTS = {".yml", ".yaml"}
SCRIPT_EXTS = {".sh", ".bash", ".ps1", ".bat", ".cmd", ".py"}
EXCLUDE_EXTS = {".gradle", ".kts"}                 # exclude any Gradle/KTS build scripts

# ---- Instrumentation signals in YAML/support scripts ----
# Keys are "reasons" that will be joined in the output.
SIGNAL_PATTERNS = {
    # Generic Gradle connected test task from scripts/CI
    "gradle_connectedAndroidTest": r"(gradle|gradlew|\.\/gradlew)\s+[^\n\r]*connected.*AndroidTest",
    # Managed devices Gradle task name invocation (e.g., pixel6api33DebugAndroidTest)
    "gradle_managed_device_task": r"\b\w+(Debug|Release)?AndroidTest\b",
    # Direct ADB instrumentation
    "adb_instrument": r"\badb\s+shell\s+am\s+instrument\b",
    # Emulator/AVD provisioning via command line
    "avdmanager_or_sdkmanager": r"\b(avdmanager|sdkmanager)\b.*(system-images|--install)",
    "emulator_launch": r"\bemulator\b.*-avd\b",
    # GitHub Actions emulator runner
    "gh_actions_emulator_runner": r"reactivecircus\/android-emulator-runner",
    # Firebase Test Lab from CI
    "gcloud_ftl": r"\bgcloud\b.*\bfirebase\s+test\s+android\s+run\b",
    # Provider hints
    "browserstack": r"\bbrowserstack\b|\bbstack\b|\bapp-automate\b",
    "sauce_labs": r"\bsaucectl\b|\bsauce\s+labs\b",
    "bitbar": r"\bbitbar\b|\bsmartbear\b",
    # Orchestrator via args in scripts
    "orchestrator_flag": r"\buseOrchestrator\b|\b--use-orchestrator\b",
    # Android test runner args passed via CLI
    "runner_args": r"\b-e\s+(class|package|annotation)\b",
}

# ---- CI platform inference (best-effort) ----
PLATFORM_GUESSERS = [
    ("github_actions", r"\.github[\\/]+workflows|uses:\s|run:\s|reactivecircus\/android-emulator-runner"),
    ("circleci", r"\.circleci[\\/]|orbs:|circleci\/android"),
    ("travis", r"\.travis\.yml|^language:\s+android"),
    ("gitlab", r"\.gitlab-ci\.yml|gitlab-ci"),
    ("azure_pipelines", r"azure[-_]pipelines\.yml|pool:\s+vmImage"),
    ("bitrise", r"bitrise\.yml|bitrise-step"),
]

def safe_read_text(path: Path) -> str:
    try:
        # fast size guard
        if path.stat().st_size > MAX_BYTES:
            return ""
    except Exception:
        pass
    for enc in ("utf-8", "utf-16", "latin-1"):
        try:
            return path.read_text(encoding=enc, errors="ignore")
        except Exception:
            continue
    return ""

def is_yaml_or_support_script(p: Path) -> bool:
    if not p.is_file():
        return False
    name = p.name.lower()
    # exclude obvious gradle/kts even if someone named them oddly
    if name.endswith(".gradle") or name.endswith(".gradle.kts"):
        return False
    ext = p.suffix.lower()
    if ext in YAML_EXTS or ext in SCRIPT_EXTS:
        return True
    # allow common CI filenames without extension edge-cases (rare)
    bare = name in {"makefile"}  # add more if you need
    return bare

def repo_key_from_filename(fname: str) -> str:
    """Expect '<full_name>__Type++filename'; repo key is part before '__'."""
    base = Path(fname).name
    key = base.split("__", 1)[0] if "__" in base else Path(base).stem
    return key.strip().lower()

def file_display_name(fname: str) -> str:
    """Return the file name portion after '++' if present, else the base name."""
    base = Path(fname).name
    if "++" in base:
        return base.split("++", 1)[1]
    # fallback to everything after '__' if available
    if "__" in base:
        return base.split("__", 1)[1]
    return base

def pick_input_csv() -> Path:
    if not SUPPORT_DIR.exists():
        raise FileNotFoundError(f"SUPPORT_DIR does not exist: {SUPPORT_DIR}")
    if INPUT_CSV_NAME:
        cand = SUPPORT_DIR / INPUT_CSV_NAME
        if cand.exists():
            return cand
        print(f"[WARN] '{INPUT_CSV_NAME}' not found in {SUPPORT_DIR}. Auto-detecting...")
    csvs = sorted(SUPPORT_DIR.glob("*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in: {SUPPORT_DIR}")
    # Prefer Check.csv if present
    for p in csvs:
        if p.name.lower() == "check.csv":
            return p
    return csvs[0]

def get_full_name_column(df: pd.DataFrame) -> str:
    for c in ("full_name", "repo", "owner_repo"):
        if c in df.columns:
            return c
    raise KeyError(f"Required column not found. Tried 'full_name', 'repo', 'owner_repo'. Available: {list(df.columns)}")

def index_yaml_and_scripts_by_repo(root: Path) -> Dict[str, List[Path]]:
    idx: Dict[str, List[Path]] = {}
    if not root.exists():
        return idx
    for p in root.rglob("*"):
        if not is_yaml_or_support_script(p):
            continue
        key = repo_key_from_filename(p.name)
        idx.setdefault(key, []).append(p)
    return idx

def guess_ci_platform(path: Path, text: str) -> str:
    src = f"{str(path)}\n{text[:2000]}"  # path + first 2KB for hinting
    for platform, regex in PLATFORM_GUESSERS:
        if re.search(regex, src, re.IGNORECASE):
            return platform
    return "other_or_unknown"

def scan_file_for_signals(path: Path, compiled_patterns: Dict[str, re.Pattern]) -> Set[str]:
    txt = safe_read_text(path)
    if not txt:
        return set()
    hits: Set[str] = set()
    for name, pat in compiled_patterns.items():
        if pat.search(txt):
            hits.add(name)
    return hits

def main():
    try:
        input_csv = pick_input_csv()
        print(f"[INFO] Using CSV: {input_csv}")

        # Load repos to consider
        df = pd.read_csv(input_csv)
        name_col = get_full_name_column(df)

        # Map repo-key -> canonical full_name (preserve original case)
        repo_names: Dict[str, str] = {}
        for v in df[name_col].astype(str).tolist():
            repo_names[v.strip().lower()] = v

        if not CONFIG_DIR.exists():
            print(f"[WARN] CONFIG_DIR does not exist: {CONFIG_DIR}")

        print("[INFO] Indexing YAML + support scripts in All_Config_Files ...")
        config_idx = index_yaml_and_scripts_by_repo(CONFIG_DIR)
        print(f"[INFO] Repos with YAML/support scripts: {len(config_idx)}")

        compiled = {k: re.compile(v, re.IGNORECASE | re.DOTALL) for k, v in SIGNAL_PATTERNS.items()}

        rows: List[Dict[str, str]] = []
        total_files = 0
        flagged_files = 0

        for repo_key, paths in config_idx.items():
            if repo_key not in repo_names:
                # Skip files for repos not in the support CSV
                continue
            full_name = repo_names[repo_key]
            for p in paths:
                total_files += 1
                reasons = scan_file_for_signals(p, compiled)
                detection = "yes" if reasons else "no"
                if reasons:
                    flagged_files += 1
                ci_guess = guess_ci_platform(p, "")  # we also consider path; text already used in patterns
                rows.append({
                    "full_name": full_name,
                    "File_name": file_display_name(p.name),
                    "ci_platform_guess": ci_guess,
                    "detection": detection,
                    "detection_reason": "; ".join(sorted(reasons))
                })

        # Save results
        OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(rows, columns=["full_name", "File_name", "ci_platform_guess", "detection", "detection_reason"]) \
          .to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

        print(f"[DONE] Wrote {len(rows)} rows to: {OUTPUT_CSV}")
        print(f"[STATS] total_files_scanned={total_files}, flagged_files={flagged_files}")

    except Exception as e:
        print("\n[ERROR] Scan aborted.")
        print(f"Reason: {type(e).__name__}: {e}")
        try:
            import traceback; traceback.print_exc()
        except Exception:
            pass

if __name__ == "__main__":
    main()


[INFO] Using CSV: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\support\Check_Y1_B0_A0.csv
[INFO] Indexing YAML + support scripts in All_Config_Files ...
[INFO] Repos with YAML/support scripts: 4519
[DONE] Wrote 171 rows to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\support\Check_Y1_B0_A0_output.csv
[STATS] total_files_scanned=171, flagged_files=19


In [ ]:
#check Y1_B0_A1 

# -*- coding: utf-8 -*-
"""
Scan YAML CI files and Gradle build files for instrumentation-testing signals.
Outputs a per-file CSV with:
  full_name, File_name, YML_check, YML_reason, Build_Check, Build_reason

Paths:
- Input CSV is read from SUPPORT_DIR (auto-detects the most recent .csv if Check.csv not found)
- Config files scanned under CONFIG_DIR
- Output CSV written to SUPPORT_DIR / "Check_Y1_B0_A1.csv"
"""

import re
import csv
from pathlib import Path
from typing import Dict, List, Tuple, Set
import pandas as pd

# ---------- PATHS (edit if needed) ----------
SUPPORT_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\support")
INPUT_CSV_NAME = "Check.csv"  # leave as-is; script will auto-detect if missing
CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files")
OUTPUT_NAME = "Check_Y1_B0_A1.csv"
# -------------------------------------------

# ---------- FILE FILTERS ----------
YAML_EXTS = {".yml", ".yaml"}
GRADLE_FILE_NAMES = (".gradle", ".gradle.kts")  # endswith checks (covers names like build.gradle, settings.gradle.kts)
# ---------------------------------

# ---------- PATTERNS: YAML / CI (raw text) ----------
YAML_SIGNAL_PATTERNS: Dict[str, str] = {
    # Gradle device/instrumentation tasks
    "gradle_connectedAndroidTest": r"(?:^|\s)(?:gradle|gradlew|\.\/gradlew)\s+[^\n\r]*\bconnectedAndroidTest\b",
    "gradle_connectedCheck":       r"(?:^|\s)(?:gradle|gradlew|\.\/gradlew)\s+[^\n\r]*\bconnectedCheck\b",
    "gradle_managed_device_task":  r"\b\w+(Debug|Release)?AndroidTest\b",  # e.g., pixel6api33DebugAndroidTest
    # Emulator / device setup
    "android_emulator_runner":     r"reactivecircus\/android-emulator-runner",
    "adb_instrument":              r"\badb\s+shell\s+am\s+instrument\b",
    # Firebase Test Lab
    "gcloud_ftl_android":          r"\bgcloud\b[^\n\r]*\bfirebase\s+test\s+android\s+run\b",
    # Detox E2E on emulator/simulator
    "detox_test":                  r"\bnpx\s+detox\s+test\b",
    "detox_build":                 r"\bnpx\s+detox\s+build\b",
}
# --------------------------------------------

# ---------- PATTERNS: Gradle build scripts (raw text) ----------
BUILD_SIGNAL_PATTERNS: Dict[str, str] = {
    # Runner
    "testInstrumentationRunner":   r"\btestInstrumentationRunner\b",
    # androidTest dependencies (string and catalog alias forms)
    "androidTestDependency_str":   r"\bandroidTest(Implementation|Api|CompileOnly|RuntimeOnly)\s*\(?\s*['\"][^'\"]+['\"]",
    "androidTestDependency_alias": r"\bandroidTest(Implementation|Api|CompileOnly|RuntimeOnly)\s*\(\s*[a-zA-Z0-9_.]+\s*\)",
    "androidTestDependency_add":   r"\badd\s*\(\s*['\"]androidTest(Implementation|Api|CompileOnly|RuntimeOnly)['\"]\s*,\s*[^)]+\)",
    # common libs
    "androidx_test_dep":           r"['\"][^'\"]*androidx\.test[^'\"]*['\"]",
    "espresso_dep":                r"['\"][^'\"]*espresso[^'\"]*['\"]",
    "uiautomator_dep":             r"['\"][^'\"]*uiautomator[^'\"]*['\"]",
    "orchestrator_dep":            r"['\"][^'\"]*androidx\.test:orchestrator[^'\"]*['\"]",
    "benchmark_dep":               r"['\"][^'\"]*androidx\.benchmark[^'\"]*['\"]",
    # Gradle config features
    "managed_devices_block":       r"\btestOptions\s*\{[^}]*managedDevices\b|testOptions\.managedDevices",
    "useOrchestrator_flag":        r"\buseOrchestrator\b",
}
# --------------------------------------------

def safe_read_text(path: Path, max_bytes: int = 2_000_000) -> str:
    try:
        if path.stat().st_size > max_bytes:
            return ""
    except Exception:
        pass
    for enc in ("utf-8", "utf-16", "latin-1"):
        try:
            return path.read_text(encoding=enc, errors="ignore")
        except Exception:
            continue
    return ""

def pick_input_csv() -> Path:
    if INPUT_CSV_NAME:
        cand = SUPPORT_DIR / INPUT_CSV_NAME
        if cand.exists():
            return cand
    # auto-pick the most recent CSV
    csvs = sorted(SUPPORT_DIR.glob("*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in {SUPPORT_DIR}")
    # prefer Check.csv if present
    for p in csvs:
        if p.name.lower() == "check.csv":
            return p
    return csvs[0]

def repo_key_from_filename(fname: str) -> str:
    base = Path(fname).name
    return (base.split("__", 1)[0] if "__" in base else Path(base).stem).strip().lower()

def file_display_name(fname: str) -> str:
    base = Path(fname).name
    if "++" in base:
        return base.split("++", 1)[1]
    if "__" in base:
        return base.split("__", 1)[1]
    return base

def index_config_files_by_repo(root: Path) -> Dict[str, List[Path]]:
    idx: Dict[str, List[Path]] = {}
    if not root.exists():
        return idx
    for p in root.rglob("*"):
        if p.is_file():
            key = repo_key_from_filename(p.name)
            idx.setdefault(key, []).append(p)
    return idx

def is_yaml(p: Path) -> bool:
    return p.suffix.lower() in YAML_EXTS

def is_gradle(p: Path) -> bool:
    name = p.name.lower()
    return name.endswith(GRADLE_FILE_NAMES)

def scan_text(text: str, patterns: Dict[str, str]) -> Set[str]:
    hits: Set[str] = set()
    if not text:
        return hits
    for name, pat in patterns.items():
        if re.search(pat, text, flags=re.IGNORECASE | re.DOTALL):
            hits.add(name)
    return hits

def main():
    input_csv = pick_input_csv()
    df = pd.read_csv(input_csv)
    if "full_name" not in df.columns:
        # allow common fallbacks
        for alt in ("repo", "owner_repo"):
            if alt in df.columns:
                df = df.rename(columns={alt: "full_name"})
                break
    if "full_name" not in df.columns:
        raise KeyError("Input CSV must have a 'full_name' column (or 'repo'/'owner_repo').")

    # Map repo_key -> canonical full_name (preserve case)
    repo_names: Dict[str, str] = {}
    for v in df["full_name"].astype(str).tolist():
        repo_names[v.strip().lower()] = v

    config_idx = index_config_files_by_repo(CONFIG_DIR)

    compiled_yaml = {k: re.compile(v, re.IGNORECASE | re.DOTALL) for k, v in YAML_SIGNAL_PATTERNS.items()}
    compiled_build = {k: re.compile(v, re.IGNORECASE | re.DOTALL) for k, v in BUILD_SIGNAL_PATTERNS.items()}

    rows: List[Dict[str, str]] = []
    for repo_key, paths in config_idx.items():
        if repo_key not in repo_names:
            continue  # only include repos listed in input CSV
        full_name = repo_names[repo_key]

        for p in paths:
            text = safe_read_text(p)
            if not text:
                # still output the row, but mark as no/NA with empty reasons
                yml_check, yml_reason = ("NA", "") if not is_yaml(p) else ("no", "")
                build_check, build_reason = ("NA", "") if not is_gradle(p) else ("no", "")
                rows.append({
                    "full_name": full_name,
                    "File_name": file_display_name(p.name),
                    "YML_check": yml_check,
                    "YML_reason": yml_reason,
                    "Build_Check": build_check,
                    "Build_reason": build_reason,
                })
                continue

            # YAML check (only for .yml/.yaml)
            if is_yaml(p):
                yaml_hits = {name for name, pat in compiled_yaml.items() if pat.search(text)}
                yml_check = "yes" if yaml_hits else "no"
                yml_reason = "; ".join(sorted(yaml_hits))
            else:
                yml_check, yml_reason = "NA", ""

            # Build check (only for Gradle files)
            if is_gradle(p):
                build_hits = {name for name, pat in compiled_build.items() if pat.search(text)}
                build_check = "yes" if build_hits else "no"
                build_reason = "; ".join(sorted(build_hits))
            else:
                build_check, build_reason = "NA", ""

            rows.append({
                "full_name": full_name,
                "File_name": file_display_name(p.name),
                "YML_check": yml_check,
                "YML_reason": yml_reason,
                "Build_Check": build_check,
                "Build_reason": build_reason,
            })

    out_path = SUPPORT_DIR / OUTPUT_NAME
    pd.DataFrame(rows, columns=["full_name", "File_name", "YML_check", "YML_reason", "Build_Check", "Build_reason"]) \
      .to_csv(out_path, index=False, encoding="utf-8")

    print(f"[DONE] Wrote {len(rows)} rows → {out_path}")

if __name__ == "__main__":
    main()


[DONE] Wrote 428 rows → C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\support\Check_Y1_B0_A1.csv


In [10]:
# for Y1_B1_A0 check & & Y1_B1_A1 
# -*- coding: utf-8 -*-
"""
Scan YAML & Gradle files for Android instrumentation signals and write results.

- Input CSV must contain a 'full_name' column.
- Matches files in All_Config_Files by filename prefix before the first "__".
- For each matched file:
    * YAML (.yml/.yaml): set YML_Check = yes/no + YML_reason
    * Gradle (.gradle/.gradle.kts): set Build_check = yes/no + Build_reason
    * Non-applicable column is set to "NA" with empty reason
- Output CSV (one row per file) includes: full_name, file_name, YML_Check, YML_reason,
  Build_check, Build_reason

Output path:
C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\support\Check_Y1_B1_A0_output.csv
C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\support\Check_Y1_B1_A1_output.csv
"""

import re
import shutil
from pathlib import Path
from typing import Dict, List, Tuple, Set

import pandas as pd

# ----------------- CONFIG -----------------
SUPPORT_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\support")
CONFIG_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files")

# If you know the exact input CSV name, set it here (e.g., "Check.csv").
# Otherwise, the script will auto-pick the most recent CSV in SUPPORT_DIR.
INPUT_CSV_NAME = "Check_Y1_B1_A1.csv"

OUTPUT_CSV_NAME = "Check_Y1_B1_A1_output.csv"
MAKE_BACKUP_OF_INPUT = False
# ------------------------------------------


# ---- YAML (CI) instrumentation/test signals (raw-text) ----
YAML_SIGNAL_PATTERNS = {
    # Android emulator runner action
    "emulator_runner_action": r"uses:\s*reactivecircus/android-emulator-runner@",
    # Gradle connected tests
    "connected_check_task": r"\bconnected(AndroidTest|Check)\b",
    "gradle_connected_task": r"\bgradle[w]?\b[^\n\r]*\bconnected(AndroidTest|Check)\b",
    # Managed devices: tasks like <device><BuildType>AndroidTest (when invoked)
    "managed_device_androidtest_task": r"\b[a-zA-Z0-9_]+(Debug|Release)?AndroidTest\b",
    # Direct instrumentation via ADB
    "adb_instrument": r"\badb\s+shell\s+am\s+instrument\b",
    # Firebase Test Lab
    "gcloud_ftl": r"\bgcloud\b[^\n\r]*\bfirebase\s+test\s+android\s+run\b",
    # Detox (Android)
    "detox_android": r"\bnpx\s+detox\b[^\n\r]*(android|--configuration\s+android\.)",
    # AVD creation / emulator boot via CLI
    "avd_create": r"\b(avdmanager|android\s+create\s+avd)\b",
    "emulator_launch": r"\bemulator\s+-avd\b",
    # Flutter tests explicitly targeting Android device/emulator
    "flutter_android_device_test": r"\bflutter\s+test\b[^\n\r]*\b-d\b[^\n\r]*(emulator|android)",
}

# ---- Gradle/KTS instrumentation build signals ----
BUILD_SIGNAL_PATTERNS = {
    # Runner
    "testInstrumentationRunner": r"\btestInstrumentationRunner\b\s*(=|\s)\s*['\"][^'\"]+['\"]?",
    # androidTest dependency (string notation)
    "androidTestDependency_str": r"\bandroidTest(Implementation|Api|CompileOnly|RuntimeOnly)\s*\(?\s*['\"][^'\"]+['\"]",
    # androidTest dependency (catalog/alias notation)
    "androidTestDependency_alias": r"\bandroidTest(Implementation|Api|CompileOnly|RuntimeOnly)\s*\(\s*[a-zA-Z0-9_.:-]+\s*\)",
    # androidTest dependency via add("androidTestImplementation", ...)
    "androidTestDependency_add": r"\badd\s*\(\s*['\"]androidTest(Implementation|Api|CompileOnly|RuntimeOnly)['\"]\s*,\s*[^)]+\)",
    # androidx.test / espresso / uiautomator / orchestrator / benchmark
    "androidx_test_dependency": r"['\"][^'\"]*androidx\.test[^'\"]*['\"]",
    "espresso_dependency": r"['\"][^'\"]*espresso[^'\"]*['\"]",
    "uiautomator_dependency": r"['\"][^'\"]*uiautomator[^'\"]*['\"]",
    "orchestrator_dependency": r"['\"][^'\"]*androidx\.test:orchestrator[^'\"]*['\"]",
    "benchmark_dependency": r"['\"][^'\"]*androidx\.benchmark[^'\"]*['\"]",
    # Managed devices / orchestrator usage
    "managed_devices": r"\btestOptions\s*\{[^}]*managedDevices\b|testOptions\.managedDevices",
    "useOrchestrator": r"\buseOrchestrator\s*(=|\s)\s*true\b",
    # CI tasks sometimes mentioned in build scripts
    "connectedAndroidTest_task": r"\bconnectedAndroidTest\b",
}

YAML_EXTS = {".yml", ".yaml"}
GRADLE_EXTS = {".gradle", ".gradle.kts"}


def safe_read_text(p: Path) -> str:
    for enc in ("utf-8", "latin-1"):
        try:
            return p.read_text(encoding=enc, errors="ignore")
        except Exception:
            continue
    return ""


def repo_key_from_filename(fname: str) -> str:
    """
    Extract repo key from saved filename '<full_name>__...'.
    """
    base = Path(fname).name
    key = base.split("__", 1)[0] if "__" in base else Path(base).stem
    return key.strip().lower()


def pick_input_csv() -> Path:
    if INPUT_CSV_NAME:
        p = SUPPORT_DIR / INPUT_CSV_NAME
        if p.exists():
            return p
        print(f"[WARN] '{INPUT_CSV_NAME}' not found, falling back to auto-detect.")
    csvs = sorted(SUPPORT_DIR.glob("*.csv"), key=lambda x: x.stat().st_mtime, reverse=True)
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in: {SUPPORT_DIR}")
    # Prefer a CSV named 'Check.csv' if present; else most recent
    for c in csvs:
        if c.name.lower() == "check.csv":
            return c
    return csvs[0]


def index_repo_files(root: Path) -> Dict[str, List[Path]]:
    """
    Index *all* candidate files (.yml/.yaml/.gradle/.gradle.kts) by repo key.
    """
    idx: Dict[str, List[Path]] = {}
    if not root.exists():
        return idx
    for p in root.rglob("*"):
        if p.is_file() and (p.suffix.lower() in YAML_EXTS or p.name.endswith(".gradle.kts") or p.suffix.lower() == ".gradle"):
            key = repo_key_from_filename(p.name)
            idx.setdefault(key, []).append(p)
    return idx


def scan_text(text: str, patterns: Dict[str, str]) -> Set[str]:
    out: Set[str] = set()
    for name, pat in patterns.items():
        if re.search(pat, text, flags=re.IGNORECASE | re.DOTALL):
            out.add(name)
    return out


def main():
    # --- Load input CSV ---
    input_csv = pick_input_csv()
    print(f"[INFO] Using input CSV: {input_csv}")

    if MAKE_BACKUP_OF_INPUT:
        backup = input_csv.with_name(input_csv.stem + "__backup" + input_csv.suffix)
        shutil.copy2(input_csv, backup)
        print(f"[INFO] Backup saved: {backup}")

    df_in = pd.read_csv(input_csv)
    if "full_name" not in df_in.columns:
        raise KeyError("Input CSV must contain a 'full_name' column.")

    # --- Index files by repo key ---
    file_index = index_repo_files(CONFIG_DIR)
    print(f"[INFO] Repos indexed: {len(file_index)}")

    # --- Build output rows ---
    rows: List[dict] = []

    for _, row in df_in.iterrows():
        full_name_raw = str(row["full_name"]).strip()
        repo_key = full_name_raw.lower()

        paths = file_index.get(repo_key, [])
        if not paths:
            # No matching files for this repo; skip (or emit a placeholder row if you prefer)
            continue

        for path in paths:
            text = safe_read_text(path)
            if not text:
                yml_check, yml_reason = ("NA", "")
                build_check, build_reason = ("NA", "")
            else:
                ext = path.suffix.lower()
                is_gradle = (ext == ".gradle") or path.name.endswith(".gradle.kts")
                is_yaml = ext in YAML_EXTS

                # YAML scan
                if is_yaml:
                    yaml_hits = scan_text(text, YAML_SIGNAL_PATTERNS)
                    yml_check = "yes" if yaml_hits else "no"
                    yml_reason = "; ".join(sorted(yaml_hits))
                else:
                    yml_check, yml_reason = ("NA", "")

                # Gradle scan
                if is_gradle:
                    build_hits = scan_text(text, BUILD_SIGNAL_PATTERNS)
                    build_check = "yes" if build_hits else "no"
                    build_reason = "; ".join(sorted(build_hits))
                else:
                    build_check, build_reason = ("NA", "")

            rows.append({
                "full_name": full_name_raw,
                "file_name": path.name,
                "YML_Check": yml_check,
                "YML_reason": yml_reason,
                "Build_check": build_check,
                "Build_reason": build_reason,
            })

    # --- Save output CSV ---
    out_path = SUPPORT_DIR / OUTPUT_CSV_NAME
    out_df = pd.DataFrame(rows, columns=[
        "full_name", "file_name", "YML_Check", "YML_reason", "Build_check", "Build_reason"
    ])
    out_df.to_csv(out_path, index=False, encoding="utf-8")
    print(f"[DONE] Wrote: {out_path}  (rows: {len(out_df)})")


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"[ERROR] {type(e).__name__}: {e}")


[INFO] Using input CSV: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\support\Check_Y1_B1_A1.csv
[INFO] Repos indexed: 4519
[DONE] Wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\support\Check_Y1_B1_A1_output.csv  (rows: 666)
